# 🧪 Statistical Tests — Worksheet
**Name:** _______________________   **Date:** _______________________

---
**How this works:**
- Each task gives you a question and a starting line or two
- You write the logic, the test, and the visualisation
- 💡 hints tell you *what* to use — not *how*
- Every chart needs a proper title, xlabel, ylabel — your job

```
p ≤ 0.05 → Reject H₀    |    p > 0.05 → Keep H₀
```
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
np.random.seed(42)
print('Ready')

---
## 📦 Dataset — Build It

Create a DataFrame called `df` with **200 student records** using these specs:

| Column | Details |
|--------|---------|
| `gender` | Male / Female, random |
| `study_hours` | Normal(mean=4, std=1.2), clipped 0–10 |
| `sleep_hours` | Normal(mean=7, std=1.0), clipped 4–10 |
| `marks` | `study*6 + sleep*2 + noise(0,8)`, clipped 0–100 |
| `subject` | Maths 40%, Science 35%, Arts 25% |
| `passed` | 'Yes' if marks ≥ 50 else 'No' |
| `school` | School A / B / C / D — equal chance |

After creating marks, add a school boost: A:+5, B:0, C:−5, D:+10 (clip to 100).

💡 `np.random.choice` · `np.random.normal` · `.clip()` · `np.where` · `pd.DataFrame`

In [ ]:
n = 200

# Gender
gender = np.random.choice(['Male', 'Female'], size=n)

# Study hours
study_hours = np.clip(np.random.normal(4, 1.2, n), 0, 10)

# Sleep hours
sleep_hours = np.clip(np.random.normal(7, 1.0, n), 4, 10)

# Subject
subject = np.random.choice(['Maths', 'Science', 'Arts'], size=n, p=[0.4, 0.35, 0.25])

# School
school = np.random.choice(['School A', 'School B', 'School C', 'School D'], size=n)

# Base marks
marks = study_hours * 6 + sleep_hours * 2 + np.random.normal(0, 8, n)

# School boost
boost = {'School A': 5, 'School B': 0, 'School C': -5, 'School D': 10}
marks = marks + pd.Series(school).map(boost)

# Clip marks
marks = np.clip(marks, 0, 100)

# Passed
passed = np.where(marks >= 50, 'Yes', 'No')

# DataFrame
df = pd.DataFrame({
    'gender': gender,
    'study_hours': study_hours,
    'sleep_hours': sleep_hours,
    'marks': marks,
    'subject': subject,
    'passed': passed,
    'school': school
})

print(df.shape)
df.head()

---
## Task 1 — Explore the Data (Visualisation)

Before any test, look at your data. Create a **2×2 grid of subplots**:
- `[0,0]` Histogram of `marks`
- `[0,1]` Histogram of `study_hours`
- `[1,0]` Boxplot of `marks` by `gender`
- `[1,1]` Boxplot of `marks` by `school`

Label everything. Write one observation below each plot as a comment.

💡 `plt.subplots(2,2)` · `ax.hist()` · `df.boxplot(column, by, ax=)`

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Histogram marks
axes[0,0].hist(df['marks'], bins=20)
axes[0,0].set_title('Distribution of Marks')
axes[0,0].set_xlabel('Marks')
axes[0,0].set_ylabel('Frequency')

# Histogram study hours
axes[0,1].hist(df['study_hours'], bins=20)
axes[0,1].set_title('Study Hours Distribution')
axes[0,1].set_xlabel('Study Hours')
axes[0,1].set_ylabel('Frequency')

# Boxplot gender
df.boxplot(column='marks', by='gender', ax=axes[1,0])
axes[1,0].set_title('Marks by Gender')
axes[1,0].set_xlabel('Gender')
axes[1,0].set_ylabel('Marks')

# Boxplot school
df.boxplot(column='marks', by='school', ax=axes[1,1])
axes[1,1].set_title('Marks by School')
axes[1,1].set_xlabel('School')
axes[1,1].set_ylabel('Marks')

plt.tight_layout()
plt.show()

---
## Task 2 — Normality Test

Test all three numeric columns (`marks`, `study_hours`, `sleep_hours`) for normality.

- Loop over the columns
- Run Shapiro-Wilk for each
- Print: column name, p-value, and verdict
- Draw Q-Q plots for all three in a 1×3 row

💡 `stats.shapiro()` · `stats.probplot(col, plot=ax)`

In [ ]:
cols = ['marks', 'study_hours', 'sleep_hours']

for col in cols:
    stat, p = stats.shapiro(df[col])
    print(f"{col}: p={p:.4f} → {'Normal' if p>0.05 else 'Not Normal'}")

# Q-Q plots
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for i, col in enumerate(cols):
    stats.probplot(df[col], plot=axes[i])
    axes[i].set_title(f'Q-Q Plot: {col}')

plt.tight_layout()
plt.show()

---
## Task 3 — t-test: Gender vs Marks

**Question:** Do Male and Female students score differently?

- H₀: Male and female marks have the same mean
- H₁: Means are different

Run the test, print results, write your if/else verdict, then draw a **violin plot** split by gender.

💡 `stats.ttest_ind()` · `sns.violinplot()`

In [ ]:
male = df[df['gender']=='Male']['marks']
female = df[df['gender']=='Female']['marks']

stat, p = stats.ttest_ind(male, female)
print("p-value:", p)

if p <= 0.05:
    print("Reject H0: Marks differ by gender")
else:
    print("Keep H0: No significant difference")

sns.violinplot(x='gender', y='marks', data=df)
plt.title('Marks by Gender')
plt.show()

---
## Task 4 — ANOVA + Post-hoc: Schools vs Marks

**Question:** Do all 4 schools score the same?

- Run ANOVA first — if significant, run Tukey HSD to find which pairs differ
- Print ANOVA result + Tukey summary
- Draw a **bar chart with error bars** (mean ± std per school)

💡 `stats.f_oneway(*groups)` · `pairwise_tukeyhsd(endog, groups, alpha=0.05)` · `ax.bar(yerr=)`

In [ ]:
groups = [df[df['school']==s]['marks'] for s in df['school'].unique()]

stat, p = stats.f_oneway(*groups)
print("ANOVA p:", p)

tukey = pairwise_tukeyhsd(df['marks'], df['school'])
print(tukey)

# Bar plot
means = df.groupby('school')['marks'].mean()
stds = df.groupby('school')['marks'].std()

means.plot(kind='bar', yerr=stds)
plt.title('Marks by School')
plt.ylabel('Marks')
plt.show()

---
## Task 5 — Mann-Whitney U: Study Hours by Gender

Study hours are likely skewed — use a non-parametric test.

**Question:** Do male and female students study the same number of hours?

- Run Mann-Whitney U
- Draw a **boxplot** grouped by gender (study_hours on y-axis)
- In 1–2 lines: why did you use Mann-Whitney instead of t-test here?

💡 `stats.mannwhitneyu(g1, g2, alternative='two-sided')`

In [ ]:
male = df[df['gender']=='Male']['study_hours']
female = df[df['gender']=='Female']['study_hours']

stat, p = stats.mannwhitneyu(male, female)
print("p:", p)

sns.boxplot(x='gender', y='study_hours', data=df)
plt.title('Study Hours by Gender')
plt.show()

---
## Task 6 — Pearson + Spearman: Study Hours vs Marks

Run **both** Pearson and Spearman on (study_hours, marks).

- Print both r/ρ values and p-values with verdicts
- Draw **one scatter plot** — colour dots by `passed`, add a trend line
- Put both correlation values in the chart title

💡 `stats.pearsonr` · `stats.spearmanr` · `sns.scatterplot(hue=)` · `np.polyfit`

In [ ]:
# Both tests + scatter with trend line







# Do Pearson and Spearman agree? What does that tell you?
# Answer:

---
## Task 7 — Correlation Heatmap with Significance Mask

1. Compute the full correlation matrix for the 3 numeric columns
2. Build a p-value matrix using a nested loop (Pearson for each pair)
3. Create a **mask** where p > 0.05
4. Draw **two heatmaps side by side** — full matrix vs masked (significant only)

💡 `df.corr()` · nested loop with `stats.pearsonr` · `sns.heatmap(mask=)`

In [ ]:
num_cols = ['marks', 'study_hours', 'sleep_hours']

# Correlation matrix


# p-value matrix (nested loop)




# Mask


# Two heatmaps side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))




plt.tight_layout()
plt.show()

---
## Task 8 — Chi-Square: Subject vs Passed

**Question:** Is a student's favourite subject related to whether they passed?

- Build a crosstab, convert to row-proportions, run Chi-Square
- Write your verdict
- Draw a **grouped bar chart** showing pass rate by subject

💡 `pd.crosstab` · `.div(axis=0)` · `stats.chi2_contingency` · `DataFrame.plot(kind='bar')`

In [ ]:
# Crosstab, proportion table, chi-square, verdict, grouped bar chart







# Which subject has the highest pass rate?
# Answer:

---
## Task 9 — 🎓 The Cheating Story

A student scored **85**. The class uses your `df` marks. The teacher is suspicious.

**Part A:** Run a one-sample t-test. Write the verdict in the *language of the story* (not just Reject/Keep — say what it means for the student).

**Part B — Sensitivity:** Loop scores 55 to 100. For each, compute the p-value. Plot **score vs p-value** with the α line. Mark the threshold where suspicion begins.

💡 `stats.ttest_1samp(data, popmean=)` · `np.arange` · `ax.axhline` · `ax.axvline`

In [ ]:
# Part A




# Part B — loop + sensitivity plot






# At what score does the test become significant?
# Answer:

---
## Task 10 — Parametric vs Non-Parametric on Skewed Data

Create a **skewed version** of marks: `np.where(marks > 80, marks**1.4, marks)`, clip to 100.

Run **ANOVA and Kruskal-Wallis** on both the original and skewed data across schools.

Show all 4 results in a **2×2 subplot** (original vs skewed × ANOVA vs Kruskal).

Then answer: which test is more reliable on skewed data, and why?

💡 `stats.f_oneway` · `stats.kruskal` · `sns.boxplot` in each subplot

In [ ]:
# Skewed version
marks_skewed = np.clip(np.where(df['marks'] > 80, df['marks'] ** 1.4, df['marks']), 0, 100)
df['marks_skewed'] = marks_skewed

# Run all 4 tests and store p-values




# 2×2 subplot
fig, axes = plt.subplots(2, 2, figsize=(13, 8))





plt.tight_layout()
plt.show()

# Which test is more reliable on skewed data?
# Answer:

---
## Task 11 — Your Own Question

Pick **any relationship** in the dataset that hasn't been tested yet.  
Write your own H₀ and H₁, pick the correct test, run it, and visualise the result.

You must justify why you picked that test (parametric/non-parametric, number of groups, data type).

In [ ]:
# H₀:
# H₁:
# Test chosen:   Justification:

# Your code:





# Verdict:

---
## Task 12 — Summary Dashboard

Collect all your p-values from Tasks 2–10 into a list of dicts.  
Convert to a DataFrame and print the table.  
Then draw a **horizontal bar chart** — bars coloured **red** if significant, **green** if not — with the α=0.05 line.

💡 `pd.DataFrame(list_of_dicts)` · `ax.barh()` · `ax.axvline(0.05)`

In [ ]:
# Build results list — fill in your p-values from earlier tasks
results = [
    {'Test': 'Shapiro-Wilk — marks',          'p_value': },
    {'Test': 'Shapiro-Wilk — study_hours',     'p_value': },
    {'Test': 't-test — gender vs marks',       'p_value': },
    {'Test': 'ANOVA — school vs marks',        'p_value': },
    {'Test': 'Mann-Whitney — gender vs study', 'p_value': },
    {'Test': 'Pearson — study vs marks',       'p_value': },
    {'Test': 'Spearman — sleep vs marks',      'p_value': },
    {'Test': 'Chi-Square — subject vs passed', 'p_value': },
    {'Test': 'One-sample t — cheating',        'p_value': },
]

# Add 'significant' and 'verdict' columns, print table, draw bar chart







---
## ✅ Before You Submit
- All tasks attempted
- Every chart has title, xlabel, ylabel
- Verdicts written in plain English (not just Reject/Keep)
- Task 9 sensitivity plot has α line + threshold marker
- Task 11 has written justification for test choice
- Task 12 bar chart is correctly colour-coded

---
*Statistics doesn't prove truth — it measures surprise. 🎓*